# CARE-BED — single W20_HSG model for the fresh live experiment

This notebook keeps the original CARE-BED BiLSTM training workflow as close as possible to the notebook used for the
original paper, while correcting the methodological problems that caused data leakage.

## What stays the same

```text
BiLSTM(64)
Dropout(0.5)
Dense softmax output
Adam, learning rate = 0.0005
Sparse categorical cross-entropy
batch size = 8
maximum 50 epochs
EarlyStopping on validation loss
patience = 5
restore_best_weights = True
20-frame CSI input
54 retained amplitude features
Hampel + Savitzky-Golay preprocessing
stride = 5
```

## What is corrected

The original notebook:

```text
created overlapping W20 segments
→ fitted StandardScaler on all segments
→ random segment-level train/test split
→ validation_split=0.2 inside the training segments
```

That allowed overlapping windows from the same continuous recording to appear in different partitions.

This notebook instead performs the split **before segmentation**:

```text
25 independent recordings
        ↓
fixed recording-disjoint 80/20 split
        ↓
20 DEVELOPMENT recordings
5 completely unseen TEST recordings
```

For each DEVELOPMENT recording:

```text
first 80% of raw frames → TRAIN
last 20% of raw frames  → VALIDATION
```

Then TRAIN, VALIDATION, and TEST are preprocessed and segmented independently.

The scaler is fitted on TRAIN only.

The resulting single model is evaluated on the five held-out recordings and **the same trained model is saved for the
fresh live test**. There is no second retraining step.

The separate 25-fold LORO experiment remains the stronger analysis of recording-to-recording generalisation and
variability. This single fixed split is the deployment-model training experiment, analogous to the workflow used in the
original paper but without train/test leakage.

## 0. Install packages

In [ ]:
!pip -q install hampel joblib

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
from math import sqrt
import csv
import json
import random
import re
import shutil
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hampel import hampel
from scipy.signal import savgol_filter

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Upload the two datasets

Upload:

```text
envA_dataset.zip
carebed_new_data.zip
```

Environment B uses `s01`, `s02`, and `s03` for each of the four activities, matching the final A13+B12 experiments.

In [ ]:
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()

OLD_ZIP = WORKDIR / "envA_dataset.zip"
NEW_ZIP = WORKDIR / "carebed_new_data.zip"

if not OLD_ZIP.exists() or not NEW_ZIP.exists():
    try:
        from google.colab import files
        print("Upload envA_dataset.zip and carebed_new_data.zip")
        files.upload()
    except ImportError:
        pass

if not OLD_ZIP.exists():
    raise FileNotFoundError("Missing envA_dataset.zip")

if not NEW_ZIP.exists():
    raise FileNotFoundError("Missing carebed_new_data.zip")

OLD_ROOT = WORKDIR / "envA_dataset"
NEW_ROOT = WORKDIR / "carebed_new_data"

for root in [OLD_ROOT, NEW_ROOT]:
    if root.exists():
        shutil.rmtree(root)

with zipfile.ZipFile(OLD_ZIP, "r") as zf:
    zf.extractall(WORKDIR)

with zipfile.ZipFile(NEW_ZIP, "r") as zf:
    zf.extractall(WORKDIR)

if not OLD_ROOT.exists():
    raise FileNotFoundError(
        "envA_dataset.zip must contain top-level envA_dataset/"
    )

if not NEW_ROOT.exists():
    raise FileNotFoundError(
        "carebed_new_data.zip must contain top-level carebed_new_data/"
    )

print("Datasets extracted.")

## 3. Configuration

In [ ]:
EXPECTED_ROLE = "AP"
NEW_MAC = "24:0A:C4:02:29:F0"

OLD_SOURCE_TOKENS = {
    "inactivity": "br_ak",
    "lying down": "lezenie",
    "sitting up": "siedzenie",
    "fidgeting": "wiercenie",
}

OLD_RECORDING_RANGES = {
    "inactivity": [
        ("A_inactivity_s01", 1, 1765),
        ("A_inactivity_s02", 1766, 2427),
        ("A_inactivity_s03", 2428, 3095),
        ("A_inactivity_s04", 3096, 3202),
    ],
    "lying down": [
        ("A_lying_down_s01", 1, 607),
        ("A_lying_down_s02", 608, 1779),
        ("A_lying_down_s03", 1780, 2947),
    ],
    "sitting up": [
        ("A_sitting_up_s01", 1, 601),
        ("A_sitting_up_s02", 602, 1739),
        ("A_sitting_up_s03", 1740, 2922),
    ],
    "fidgeting": [
        ("A_fidgeting_s01", 1, 1123),
        ("A_fidgeting_s02", 1124, 1810),
        ("A_fidgeting_s03", 1811, 2955),
    ],
}

EXPECTED_OLD_LINE_COUNTS = {
    "inactivity": 3202,
    "lying down": 2947,
    "sitting up": 2922,
    "fidgeting": 2955,
}

B_FOLDERS = {
    "inactivity": "inactivity",
    "lying down": "lying_down",
    "sitting up": "sitting_up",
    "fidgeting": "fidgeting",
}

VALID_B_SESSIONS = {1, 2, 3}

FIRST_CSI_VALUES = 128
RAW_FEATURES = 64

COLUMNS_TO_DROP = [2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
N_FEATURES = 54

WINDOW_LENGTH = 20
STRIDE = 5

# Approximate the original 80/20 test proportion,
# but split by complete recordings instead of overlapping segments.
TEST_RECORDINGS = 5

# Validation remains approximately 20%, but is created within each
# development recording BEFORE filtering and segmentation.
DEV_TRAIN_FRACTION = 0.80

HAMPEL_WINDOW = 10
SAVGOL_WINDOW = 10
SAVGOL_POLYORDER = 3

LEARNING_RATE = 0.0005
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 5

MODEL_FILE = WORKDIR / "carebed_W20_HSG_single_split_live_model.keras"
SCALER_FILE = WORKDIR / "carebed_W20_HSG_single_split_scaler.pkl"
ENCODER_FILE = WORKDIR / "carebed_W20_HSG_single_split_label_encoder.pkl"
METADATA_FILE = WORKDIR / "carebed_W20_HSG_single_split_metadata.json"

DATASET_SUMMARY_CSV = WORKDIR / "carebed_W20_HSG_dataset_summary.csv"
SPLIT_CSV = WORKDIR / "carebed_W20_HSG_recording_split.csv"
HISTORY_CSV = WORKDIR / "carebed_W20_HSG_training_history.csv"
TEST_PREDICTIONS_CSV = WORKDIR / "carebed_W20_HSG_test_predictions.csv"
TEST_REPORT_CSV = WORKDIR / "carebed_W20_HSG_test_classification_report.csv"
PER_RECORDING_CSV = WORKDIR / "carebed_W20_HSG_test_per_recording.csv"

LOSS_PDF = WORKDIR / "carebed_W20_HSG_learning_loss.pdf"
ACCURACY_PDF = WORKDIR / "carebed_W20_HSG_learning_accuracy.pdf"
CONFUSION_PDF = WORKDIR / "carebed_W20_HSG_test_confusion_matrix.pdf"
PER_RECORDING_PDF = WORKDIR / "carebed_W20_HSG_test_per_recording_accuracy.pdf"

RESULT_ZIP = WORKDIR / "CARE_BED_W20_HSG_SINGLE_SPLIT_LIVE_MODEL.zip"

assert RAW_FEATURES - len(COLUMNS_TO_DROP) == N_FEATURES

## 4. CSI parsing helpers

In [ ]:
def parse_bracket_payload(line):
    try:
        content = re.findall(r"\[(.*)\]", line)[0]
        return [int(x) for x in content.split(" ") if x != ""]
    except Exception:
        return None


def amplitude_from_payload(payload):
    if payload is None or len(payload) < FIRST_CSI_VALUES:
        return None

    raw = payload[:FIRST_CSI_VALUES]
    imaginary = raw[0::2]
    real = raw[1::2]

    if len(real) != RAW_FEATURES:
        return None

    return np.asarray(
        [
            sqrt(imaginary[i] ** 2 + real[i] ** 2)
            for i in range(RAW_FEATURES)
        ],
        dtype=np.float32,
    )


def old_line_to_amplitude(line):
    return amplitude_from_payload(
        parse_bracket_payload(line)
    )


def new_line_to_amplitude(line):
    try:
        row = next(csv.reader([line]))
    except Exception:
        return None

    if len(row) != 26:
        return None

    if row[0].strip() != "CSI_DATA":
        return None

    if row[1].strip() != EXPECTED_ROLE:
        return None

    if row[2].strip().upper() != NEW_MAC.upper():
        return None

    try:
        declared_len = int(row[24].strip())
    except ValueError:
        return None

    payload = parse_bracket_payload(line)

    if payload is None:
        return None

    if len(payload) != declared_len:
        return None

    return amplitude_from_payload(payload)


def session_number_from_name(path):
    match = re.search(
        r"_s(\d+)\.csv$",
        path.name,
        flags=re.IGNORECASE,
    )

    return int(match.group(1)) if match else None

## 5. Load Environment A — 13 independent recordings

In [ ]:
old_source_files = {}

for activity, token in OLD_SOURCE_TOKENS.items():
    matches = [
        path
        for path in OLD_ROOT.glob("*.csv")
        if token in path.name.lower()
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one Environment A CSV containing '{token}', "
            f"found {[p.name for p in matches]}"
        )

    old_source_files[activity] = matches[0]


recordings_A = []

for activity, ranges in OLD_RECORDING_RANGES.items():
    source = old_source_files[activity]

    with source.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as fh:
        lines = fh.readlines()

    expected = EXPECTED_OLD_LINE_COUNTS[activity]

    if len(lines) != expected:
        raise ValueError(
            f"{source.name}: expected {expected} lines, found {len(lines)}"
        )

    for recording_id, start1, end1 in ranges:
        amplitudes = [
            old_line_to_amplitude(line)
            for line in lines[start1 - 1:end1]
        ]

        amplitudes = [
            value
            for value in amplitudes
            if value is not None
        ]

        raw = pd.DataFrame(amplitudes)

        if raw.shape[1] != RAW_FEATURES:
            raise ValueError(
                f"{recording_id}: expected 64 amplitude features, got {raw.shape}"
            )

        recordings_A.append({
            "recording_id": recording_id,
            "label": activity,
            "environment": "Environment A",
            "source_file": source.name,
            "raw": raw.reset_index(drop=True),
        })

assert len(recordings_A) == 13

print("Environment A recordings:", len(recordings_A))

## 6. Load Environment B — s01, s02 and s03 only

In [ ]:
recordings_B = []

for activity, folder_name in B_FOLDERS.items():
    folder = NEW_ROOT / folder_name

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing Environment B folder: {folder}"
        )

    for path in sorted(folder.glob("*.csv")):
        if path.name.endswith("_timing.csv"):
            continue

        session_number = session_number_from_name(path)

        if session_number not in VALID_B_SESSIONS:
            continue

        amplitudes = []

        with path.open(
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as fh:
            for line in fh:
                amplitude = new_line_to_amplitude(line)

                if amplitude is not None:
                    amplitudes.append(amplitude)

        if not amplitudes:
            raise ValueError(
                f"{path.name}: no valid CSI frames parsed."
            )

        raw = pd.DataFrame(amplitudes)

        if raw.shape[1] != RAW_FEATURES:
            raise ValueError(
                f"{path.name}: expected 64 amplitude features, got {raw.shape}"
            )

        recordings_B.append({
            "recording_id": "B_" + path.stem.replace("envb_", ""),
            "label": activity,
            "environment": "Environment B",
            "source_file": path.name,
            "raw": raw.reset_index(drop=True),
        })

assert len(recordings_B) == 12

print("Environment B recordings:", len(recordings_B))

## 7. Dataset summary and total number of raw CSI frames

`raw_frames` means valid CSI frames after parsing and before filtering/segmentation.
Use the printed total when updating the dataset-size statement in the manuscript.

In [ ]:
recordings = recordings_A + recordings_B

recording_summary = pd.DataFrame([
    {
        "recording_id": r["recording_id"],
        "label": r["label"],
        "environment": r["environment"],
        "raw_frames": len(r["raw"]),
        "source_file": r["source_file"],
    }
    for r in recordings
]).sort_values(
    ["environment", "label", "recording_id"]
).reset_index(drop=True)

assert len(recordings) == 25
assert recording_summary["recording_id"].nunique() == 25

display(recording_summary)

summary_by_environment = (
    recording_summary
    .groupby(
        "environment",
        as_index=False,
    )
    .agg(
        recordings=("recording_id", "count"),
        raw_frames=("raw_frames", "sum"),
    )
)

summary_by_class = (
    recording_summary
    .groupby(
        ["environment", "label"],
        as_index=False,
    )
    .agg(
        recordings=("recording_id", "count"),
        raw_frames=("raw_frames", "sum"),
    )
)

display(summary_by_environment)
display(summary_by_class)

TOTAL_RAW_FRAMES = int(
    recording_summary["raw_frames"].sum()
)

TOTAL_A_FRAMES = int(
    recording_summary.loc[
        recording_summary["environment"] == "Environment A",
        "raw_frames",
    ].sum()
)

TOTAL_B_FRAMES = int(
    recording_summary.loc[
        recording_summary["environment"] == "Environment B",
        "raw_frames",
    ].sum()
)

print("\n" + "=" * 72)
print("FINAL A13+B12 DATASET")
print("=" * 72)
print(f"Environment A: 13 recordings | {TOTAL_A_FRAMES:,} raw CSI frames")
print(f"Environment B: 12 recordings | {TOTAL_B_FRAMES:,} raw CSI frames")
print(f"TOTAL:         25 recordings | {TOTAL_RAW_FRAMES:,} raw CSI frames")
print("=" * 72)

recording_summary.to_csv(
    DATASET_SUMMARY_CSV,
    index=False,
)

## 8. Fixed recording-disjoint 80/20 train-test split

The original notebook used `test_size=0.2`.

Here we retain approximately the same proportion but apply it to **whole recordings**:

```text
20 recordings → development
5 recordings  → completely unseen test set
```

The split is stratified by activity and fixed with `random_state=42`.

The exact recording IDs are printed and saved so that the experiment is reproducible.

In [ ]:
recording_table = recording_summary[
    [
        "recording_id",
        "label",
        "environment",
    ]
].copy()

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_RECORDINGS,
    random_state=SEED,
)

development_idx, test_idx = next(
    splitter.split(
        recording_table["recording_id"],
        recording_table["label"],
    )
)

development_ids = recording_table.iloc[
    development_idx
]["recording_id"].tolist()

test_ids = recording_table.iloc[
    test_idx
]["recording_id"].tolist()

assert set(development_ids).isdisjoint(set(test_ids))
assert len(development_ids) == 20
assert len(test_ids) == 5

split_rows = []

for _, row in recording_table.iterrows():
    split_rows.append({
        "recording_id": row["recording_id"],
        "label": row["label"],
        "environment": row["environment"],
        "outer_partition": (
            "TEST"
            if row["recording_id"] in test_ids
            else "DEVELOPMENT"
        ),
    })

split_df = pd.DataFrame(split_rows)

print("=== DEVELOPMENT RECORDINGS ===")
display(
    split_df[
        split_df["outer_partition"] == "DEVELOPMENT"
    ].sort_values(["label", "recording_id"])
)

print("\n=== HELD-OUT TEST RECORDINGS ===")
display(
    split_df[
        split_df["outer_partition"] == "TEST"
    ].sort_values(["label", "recording_id"])
)

print("\nTest-set class counts:")
display(
    split_df[
        split_df["outer_partition"] == "TEST"
    ]["label"].value_counts()
)

split_df.to_csv(
    SPLIT_CSV,
    index=False,
)

## 9. HSG preprocessing and W20 segmentation

For the 20 development recordings, each recording is divided at the **raw-frame level**:

```text
first 80% → TRAIN
last 20%  → VALIDATION
```

Those raw partitions are then independently filtered and segmented.

The five TEST recordings remain intact and are independently filtered and segmented.

This preserves the purpose of the original 20% validation set and EarlyStopping while preventing overlapping windows
from crossing train/validation/test boundaries.

In [ ]:
def preprocess_partition(df):
    processed = pd.DataFrame(
        index=df.index
    )

    for col in df.columns:
        values = df[col].astype(float)

        hampel_result = hampel(
            values,
            window_size=HAMPEL_WINDOW,
        )

        smoothed = savgol_filter(
            np.asarray(
                hampel_result.filtered_data,
                dtype=float,
            ),
            window_length=SAVGOL_WINDOW,
            polyorder=SAVGOL_POLYORDER,
        )

        processed[col] = smoothed

    processed.columns = range(
        processed.shape[1]
    )

    processed = (
        processed
        .drop(
            columns=COLUMNS_TO_DROP,
            axis=1,
        )
        .reset_index(drop=True)
    )

    if processed.shape[1] != N_FEATURES:
        raise ValueError(
            f"Expected {N_FEATURES} features, got {processed.shape[1]}"
        )

    return processed


def segment_partition(
    df,
    label,
    recording_id,
    environment,
):
    rows = []

    for start in range(
        0,
        len(df) - WINDOW_LENGTH + 1,
        STRIDE,
    ):
        end = start + WINDOW_LENGTH

        rows.append({
            "X": df.iloc[start:end].values.astype(np.float32),
            "label": label,
            "recording_id": recording_id,
            "environment": environment,
            "start_frame": start,
            "end_frame_exclusive": end,
        })

    return rows

## 10. Construct TRAIN, VALIDATION and TEST windows

In [ ]:
recordings_by_id = {
    r["recording_id"]: r
    for r in recordings
}

train_rows = []
val_rows = []
test_rows = []

partition_summary_rows = []

for recording_id in development_ids:
    recording = recordings_by_id[recording_id]
    raw = recording["raw"].reset_index(drop=True)

    cut = int(
        len(raw) * DEV_TRAIN_FRACTION
    )

    train_raw = raw.iloc[:cut].reset_index(drop=True)
    val_raw = raw.iloc[cut:].reset_index(drop=True)

    if len(train_raw) < SAVGOL_WINDOW:
        raise ValueError(
            f"{recording_id}: TRAIN partition too short for filtering."
        )

    if len(val_raw) < SAVGOL_WINDOW:
        raise ValueError(
            f"{recording_id}: VALIDATION partition too short for filtering."
        )

    train_processed = preprocess_partition(
        train_raw
    )

    val_processed = preprocess_partition(
        val_raw
    )

    recording_train_rows = segment_partition(
        train_processed,
        recording["label"],
        recording_id,
        recording["environment"],
    )

    recording_val_rows = segment_partition(
        val_processed,
        recording["label"],
        recording_id,
        recording["environment"],
    )

    train_rows.extend(
        recording_train_rows
    )

    val_rows.extend(
        recording_val_rows
    )

    partition_summary_rows.append({
        "recording_id": recording_id,
        "label": recording["label"],
        "environment": recording["environment"],
        "partition": "DEVELOPMENT",
        "raw_frames": len(raw),
        "train_raw_frames": len(train_raw),
        "validation_raw_frames": len(val_raw),
        "train_windows": len(recording_train_rows),
        "validation_windows": len(recording_val_rows),
        "test_windows": 0,
    })


for recording_id in test_ids:
    recording = recordings_by_id[recording_id]
    raw = recording["raw"].reset_index(drop=True)

    test_processed = preprocess_partition(
        raw
    )

    recording_test_rows = segment_partition(
        test_processed,
        recording["label"],
        recording_id,
        recording["environment"],
    )

    test_rows.extend(
        recording_test_rows
    )

    partition_summary_rows.append({
        "recording_id": recording_id,
        "label": recording["label"],
        "environment": recording["environment"],
        "partition": "TEST",
        "raw_frames": len(raw),
        "train_raw_frames": 0,
        "validation_raw_frames": 0,
        "train_windows": 0,
        "validation_windows": 0,
        "test_windows": len(recording_test_rows),
    })


partition_summary = pd.DataFrame(
    partition_summary_rows
)

display(
    partition_summary.sort_values(
        ["partition", "label", "recording_id"]
    )
)

print("\nWindow counts:")
print(" TRAIN:     ", len(train_rows))
print(" VALIDATION:", len(val_rows))
print(" TEST:      ", len(test_rows))

train_recording_ids = {
    row["recording_id"]
    for row in train_rows
}

val_recording_ids = {
    row["recording_id"]
    for row in val_rows
}

test_recording_ids = {
    row["recording_id"]
    for row in test_rows
}

# TRAIN and VALIDATION intentionally come from the same DEVELOPMENT recordings,
# but TEST must be completely recording-disjoint.
assert train_recording_ids == set(development_ids)
assert val_recording_ids.issubset(set(development_ids))
assert test_recording_ids == set(test_ids)
assert (
    train_recording_ids
    | val_recording_ids
).isdisjoint(
    test_recording_ids
)

print("\nRecording-disjoint TEST check — PASSED")

## 11. Label encoding and StandardScaler

Unlike the original notebook, the StandardScaler is fitted **only on TRAIN**.

The fitted scaler is then applied unchanged to VALIDATION and TEST.

In [ ]:
label_encoder = LabelEncoder()

label_encoder.fit(
    sorted(
        recording_summary["label"].unique()
    )
)

print(
    "Class order:",
    list(label_encoder.classes_),
)


def rows_to_arrays(rows):
    X = np.stack(
        [
            row["X"]
            for row in rows
        ]
    ).astype(np.float32)

    y = label_encoder.transform(
        [
            row["label"]
            for row in rows
        ]
    ).astype(np.int64)

    return X, y


X_train_raw, y_train = rows_to_arrays(
    train_rows
)

X_val_raw, y_val = rows_to_arrays(
    val_rows
)

X_test_raw, y_test = rows_to_arrays(
    test_rows
)

scaler = StandardScaler()

scaler.fit(
    X_train_raw.reshape(
        len(X_train_raw),
        -1,
    )
)


def apply_scaler(X_raw):
    X_scaled = scaler.transform(
        X_raw.reshape(
            len(X_raw),
            -1,
        )
    )

    return X_scaled.reshape(
        len(X_raw),
        WINDOW_LENGTH,
        N_FEATURES,
    ).astype(np.float32)


X_train = apply_scaler(
    X_train_raw
)

X_val = apply_scaler(
    X_val_raw
)

X_test = apply_scaler(
    X_test_raw
)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

## 12. Define the BiLSTM

Architecture and optimisation settings are retained from the original notebook.

In [ ]:
keras.backend.clear_session()

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

bilstm_model = keras.Sequential([
    keras.Input(
        shape=(
            WINDOW_LENGTH,
            N_FEATURES,
        )
    ),
    layers.Bidirectional(
        layers.LSTM(64)
    ),
    layers.Dropout(0.5),
    layers.Dense(
        len(label_encoder.classes_),
        activation="softmax",
    ),
])

bilstm_model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

bilstm_model.summary()

## 13. Train the model with EarlyStopping

This follows the original training logic:

```text
epochs = 50
batch size = 8
monitor = validation loss
patience = 5
restore best weights = True
shuffle = True
```

The only methodological change is that validation is now explicitly supplied from the leakage-safe raw-frame split
instead of Keras randomly taking 20% of already-overlapping training windows.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
)

print("Starting BiLSTM training...")

history = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(
        X_val,
        y_val,
    ),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping
    ],
    shuffle=True,
    verbose=1,
)

history_df = pd.DataFrame(
    history.history
)

history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(history_df) + 1,
    ),
)

best_epoch = int(
    np.argmin(
        history_df["val_loss"].to_numpy()
    )
    + 1
)

print("\nTraining finished.")
print("Epochs run:", len(history_df))
print("Best validation-loss epoch:", best_epoch)
print(
    "Because restore_best_weights=True, the model now contains the weights "
    "from that best validation epoch."
)

display(history_df)

history_df.to_csv(
    HISTORY_CSV,
    index=False,
)

## 14. Training and validation curves

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["loss"],
    marker="o",
    label="Training loss",
)

ax.plot(
    history_df["epoch"],
    history_df["val_loss"],
    marker="o",
    label="Validation loss",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("W20_HSG — training and validation loss")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(
    LOSS_PDF,
    bbox_inches="tight",
)
plt.show()


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["accuracy"],
    marker="o",
    label="Training accuracy",
)

ax.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="o",
    label="Validation accuracy",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.02)
ax.set_title("W20_HSG — training and validation accuracy")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(
    ACCURACY_PDF,
    bbox_inches="tight",
)
plt.show()

## 15. Evaluate the same model on the five unseen TEST recordings

These are the metrics for the **single fixed recording-disjoint split**.

They should be reported separately from the 25-fold LORO result.  
The LORO result remains the stronger generalisation estimate because every recording is held out once.

In [ ]:
test_probabilities = bilstm_model.predict(
    X_test,
    verbose=0,
)

y_pred = np.argmax(
    test_probabilities,
    axis=1,
)

test_accuracy = accuracy_score(
    y_test,
    y_pred,
)

test_macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

test_macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

test_macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

test_weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)

print("=== SINGLE FIXED RECORDING-DISJOINT TEST ===")
print(f"Accuracy:        {test_accuracy:.4f}")
print(f"Macro Precision: {test_macro_precision:.4f}")
print(f"Macro Recall:    {test_macro_recall:.4f}")
print(f"Macro F1:        {test_macro_f1:.4f}")
print(f"Weighted F1:     {test_weighted_f1:.4f}")
print()

report_text = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0,
)

print(report_text)

report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
).T

display(report_df)

report_df.to_csv(
    TEST_REPORT_CSV
)

## 16. Save held-out TEST predictions and per-recording accuracy

In [ ]:
test_metadata = pd.DataFrame([
    {
        "recording_id": row["recording_id"],
        "environment": row["environment"],
        "true_label": row["label"],
        "start_frame": row["start_frame"],
        "end_frame_exclusive": row["end_frame_exclusive"],
    }
    for row in test_rows
])

test_metadata["predicted_label"] = label_encoder.inverse_transform(
    y_pred
)

test_metadata["confidence"] = np.max(
    test_probabilities,
    axis=1,
)

test_metadata.to_csv(
    TEST_PREDICTIONS_CSV,
    index=False,
)

per_recording_rows = []

for recording_id, group in test_metadata.groupby(
    "recording_id"
):
    recording_accuracy = (
        group["true_label"]
        == group["predicted_label"]
    ).mean()

    per_recording_rows.append({
        "recording_id": recording_id,
        "environment": group["environment"].iloc[0],
        "true_activity": group["true_label"].iloc[0],
        "windows": len(group),
        "accuracy": float(recording_accuracy),
        "mean_confidence": float(group["confidence"].mean()),
    })

per_recording_df = pd.DataFrame(
    per_recording_rows
).sort_values(
    "accuracy"
).reset_index(drop=True)

display(per_recording_df)

per_recording_df.to_csv(
    PER_RECORDING_CSV,
    index=False,
)

## 17. Normalized confusion matrix — held-out TEST only

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(
        len(label_encoder.classes_)
    ),
    normalize="true",
)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
)

display(cm_df)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

image = ax.imshow(
    cm,
    vmin=0,
    vmax=1,
)

ax.set_xticks(
    range(
        len(label_encoder.classes_)
    )
)
ax.set_xticklabels(
    label_encoder.classes_,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    range(
        len(label_encoder.classes_)
    )
)
ax.set_yticklabels(
    label_encoder.classes_
)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(
    "W20_HSG — unseen recording-disjoint test set"
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            f"{cm[i, j]:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(
    image,
    ax=ax,
    label="Proportion",
)

fig.tight_layout()

fig.savefig(
    CONFUSION_PDF,
    bbox_inches="tight",
)

plt.show()

## 18. Per-recording TEST accuracy

In [ ]:
plot_df = per_recording_df.sort_values(
    "accuracy",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

positions = np.arange(
    len(plot_df)
)

ax.barh(
    positions,
    plot_df["accuracy"],
)

ax.set_yticks(
    positions
)

ax.set_yticklabels(
    plot_df["recording_id"]
)

ax.axvline(
    test_accuracy,
    linestyle="--",
    label=f"Overall test accuracy = {test_accuracy:.3f}",
)

ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_ylabel("Held-out recording")
ax.set_title(
    "Single fixed split — per-recording TEST accuracy"
)
ax.legend()

fig.tight_layout()

fig.savefig(
    PER_RECORDING_PDF,
    bbox_inches="tight",
)

plt.show()

## 19. Save exactly this trained model for the fresh live experiment

This is **the same model that produced the held-out TEST results above**.

There is no second retraining step.

In [ ]:
bilstm_model.save(
    MODEL_FILE
)

joblib.dump(
    scaler,
    SCALER_FILE,
)

joblib.dump(
    label_encoder,
    ENCODER_FILE,
)

metadata = {
    "system": "CARE-BED",
    "purpose": "single W20_HSG model for fresh live validation",
    "dataset": {
        "total_recordings": 25,
        "environment_A_recordings": 13,
        "environment_B_recordings": 12,
        "environment_B_sessions_used": [1, 2, 3],
        "total_raw_frames": TOTAL_RAW_FRAMES,
        "environment_A_raw_frames": TOTAL_A_FRAMES,
        "environment_B_raw_frames": TOTAL_B_FRAMES,
    },
    "split": {
        "strategy": "fixed recording-disjoint holdout",
        "random_state": SEED,
        "development_recordings": development_ids,
        "test_recordings": test_ids,
        "development_train_fraction_within_each_recording": DEV_TRAIN_FRACTION,
        "validation_fraction_within_each_development_recording": 1 - DEV_TRAIN_FRACTION,
    },
    "classes": list(
        label_encoder.classes_
    ),
    "input_shape": [
        WINDOW_LENGTH,
        N_FEATURES,
    ],
    "window_length": WINDOW_LENGTH,
    "offline_stride": STRIDE,
    "preprocessing": {
        "hampel": True,
        "hampel_window": HAMPEL_WINDOW,
        "savgol": True,
        "savgol_window": SAVGOL_WINDOW,
        "savgol_polyorder": SAVGOL_POLYORDER,
        "columns_dropped": COLUMNS_TO_DROP,
        "scaler_fit": "TRAIN only",
    },
    "model": {
        "architecture": "BiLSTM(64) -> Dropout(0.5) -> Dense softmax",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "maximum_epochs": MAX_EPOCHS,
        "early_stopping_patience": PATIENCE,
        "epochs_run": int(len(history_df)),
        "best_validation_loss_epoch": best_epoch,
        "restore_best_weights": True,
    },
    "single_split_test_metrics": {
        "accuracy": float(test_accuracy),
        "macro_precision": float(test_macro_precision),
        "macro_recall": float(test_macro_recall),
        "macro_f1": float(test_macro_f1),
        "weighted_f1": float(test_weighted_f1),
    },
    "interpretation": (
        "The 25-fold LORO experiment is the principal offline generalisation/variability analysis. "
        "This fixed recording-disjoint split is used to train and characterise the concrete model "
        "that is subsequently evaluated in a fresh live experiment."
    ),
}

with METADATA_FILE.open(
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        metadata,
        fh,
        indent=2,
    )

print("Saved deployment artifacts:")
print(" ", MODEL_FILE)
print(" ", SCALER_FILE)
print(" ", ENCODER_FILE)
print(" ", METADATA_FILE)

## 20. Reload sanity check

In [ ]:
loaded_model = keras.models.load_model(
    MODEL_FILE
)

loaded_scaler = joblib.load(
    SCALER_FILE
)

loaded_encoder = joblib.load(
    ENCODER_FILE
)

sample_raw = X_test_raw[:1]

sample_scaled = loaded_scaler.transform(
    sample_raw.reshape(
        1,
        -1,
    )
).reshape(
    1,
    WINDOW_LENGTH,
    N_FEATURES,
)

sample_probabilities = loaded_model.predict(
    sample_scaled,
    verbose=0,
)[0]

print("Input shape:", sample_scaled.shape)
print("Probability sum:", float(sample_probabilities.sum()))
print(
    "Predicted class:",
    loaded_encoder.inverse_transform(
        [
            int(
                np.argmax(
                    sample_probabilities
                )
            )
        ]
    )[0],
)

assert sample_scaled.shape == (
    1,
    WINDOW_LENGTH,
    N_FEATURES,
)

assert np.isclose(
    sample_probabilities.sum(),
    1.0,
    atol=1e-5,
)

print("\nReload sanity check — PASSED")

## 21. Final experiment summary

Use this cell to copy the important numbers into your notes.

In [ ]:
print("=== CARE-BED SINGLE DEPLOYMENT-MODEL EXPERIMENT ===")
print(f"Total recordings:           {len(recordings)}")
print(f"Total raw CSI frames:       {TOTAL_RAW_FRAMES:,}")
print(f"Development recordings:     {len(development_ids)}")
print(f"Test recordings:            {len(test_ids)}")
print(f"Training windows:           {len(train_rows):,}")
print(f"Validation windows:         {len(val_rows):,}")
print(f"Test windows:               {len(test_rows):,}")
print(f"Epochs run:                 {len(history_df)}")
print(f"Best val-loss epoch:        {best_epoch}")
print(f"Single-split test accuracy: {test_accuracy:.4f}")
print(f"Single-split test Macro F1: {test_macro_f1:.4f}")
print(f"Single-split Weighted F1:   {test_weighted_f1:.4f}")
print()
print("Saved model for live test:")
print(MODEL_FILE.name)
print()
print(
    "Important: use the 25-fold LORO result as the stronger offline "
    "generalisation/variability result. The metrics above describe one fixed "
    "recording-disjoint split and the concrete model used for live testing."
)

## 22. Package all artifacts

In [ ]:
files_to_package = [
    MODEL_FILE,
    SCALER_FILE,
    ENCODER_FILE,
    METADATA_FILE,
    DATASET_SUMMARY_CSV,
    SPLIT_CSV,
    HISTORY_CSV,
    TEST_PREDICTIONS_CSV,
    TEST_REPORT_CSV,
    PER_RECORDING_CSV,
    LOSS_PDF,
    ACCURACY_PDF,
    CONFUSION_PDF,
    PER_RECORDING_PDF,
]

with zipfile.ZipFile(
    RESULT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file_path in files_to_package:
        if file_path.exists():
            zf.write(
                file_path,
                arcname=file_path.name,
            )

print("Created:", RESULT_ZIP)

try:
    from google.colab import files
    files.download(
        str(RESULT_ZIP)
    )
except ImportError:
    pass